<a href="https://colab.research.google.com/github/NguyenThien1906/Image-Video-Processing/blob/main/something%20about%20testing%20more%20stuff.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!uv pip install -q --system numba-cuda==0.4.0
from numba import config
config.CUDA_ENABLE_PYNVJITLINK = 1
CALC_FLAG = 'cpu'  # 'cpu' or 'cuda'

In [4]:
from numba import prange, jit
@jit(nopython=True, parallel=True)
def conv2d_cpu(input4d, kernel4d, bias1d, output4d, stride, padding, dilation):
  for im in prange(output4d.shape[0]):
    for c_out in prange(output4d.shape[1]):
      for x in prange(output4d.shape[2]):
        for y in prange(output4d.shape[3]):
          sum = 0
          for x_k in prange(kernel4d.shape[2]):
            for y_k in prange(kernel4d.shape[3]):
              x_in = x * stride + x_k * dilation - padding
              y_in = y * stride + y_k * dilation - padding
              if x_in < 0 or y_in < 0 or x_in >= input4d.shape[2] or y_in >= input4d.shape[3]:
                # zero-padding
                continue
              else:
                for c_in in prange(input4d.shape[1]):
                  sum += input4d[im, c_in, x_in, y_in] * kernel4d[c_out, c_in, x_k, y_k]
          output4d[im, c_out, x, y] = sum + bias1d[c_out]

@jit(nopython=True, parallel=True)
def conv2d_biasBack_cpu(d_output4d, bias1d):
  for c_out in prange(bias1d.shape[0]):
    sum = 0
    for im in prange(d_output4d.shape[0]):
      for x in prange(d_output4d.shape[2]):
        for y in prange(d_output4d.shape[3]):
          sum += d_output4d[im, c_out, x, y]
    bias1d[c_out] = sum

@jit(nopython=True, parallel=True)
def flipKernel_cpu(kernel4d, output4d):
  for c_out in prange(kernel4d.shape[0]):
    for c_in in prange(kernel4d.shape[1]):
      for x_k in prange(kernel4d.shape[2]):
        for y_k in prange(kernel4d.shape[3]):
          output4d[c_out, c_in, x_k, y_k] = kernel4d[c_out, c_in, kernel4d.shape[2]-x_k-1, kernel4d.shape[3]-y_k-1]

In [9]:
import numpy as np
class NNLayer4D:
  def __init__(self, in_channels):
    self.in_channels = in_channels
    self.input = None
    self.output = None

  def forward(self, input):
    if len(input.shape) != 4:
      raise ValueError(f"Input has to be a 4D array, got {len(input.shape)} instead.")
    elif input.shape[1] != self.in_channels:
      raise ValueError(f"Input has to have {self.in_channels} channels, got {input.shape[1]} instead.")
    # to save the input for backpropagation
    self.input = input

  def backward(self, d_output):
    pass

  @classmethod
  def class_name(cls):
    pass

  def save_pickle(self):
    pass

  @classmethod
  def load_pickle(self):
    pass

class NNConv2D(NNLayer4D):
  def __init__(self, in_channels, out_channels, kernel_size, stride, padding, dilation):
    super().__init__(in_channels)
    self.out_channels = out_channels
    self.kernel_size = kernel_size
    self.stride = stride
    self.padding = padding
    self.dilation = dilation

    self.kernel = np.random.randn(out_channels, in_channels, kernel_size, kernel_size).astype(np.float32)
    self.bias = np.random.randn(out_channels).astype(np.float32)
    self.d_kernel = np.zeros_like(self.kernel)
    self.d_bias = np.zeros_like(self.bias)

  def forward(self, input):
    super().forward(input)

    output_shape = (input.shape[0], self.out_channels,
              (input.shape[2] + 2 * self.padding - self.dilation * (self.kernel_size - 1) - 1) // self.stride + 1,
              (input.shape[3] + 2 * self.padding - self.dilation * (self.kernel_size - 1) - 1) // self.stride + 1)
    self.output = np.zeros(output_shape, dtype=np.float32)

    conv2d_cpu(self.input, self.kernel, self.bias, self.output, self.stride, self.padding, self.dilation)
    return self.output

  def backward(self, d_output):
    # d(bias) = sum of each 3D image.
    conv2d_biasBack_cpu(d_output, self.d_bias)

    # d(kernel) = conv(inputT, d_outT)T
    # note: input [N, c_in, x, y], d_out [N, c_out, x, y], kernel [c_out, c_in, x, y]
    # idea: if we swap the 1st and 2nd axis, we get that d_out [c_out, N] can be used as a kernel for input [c_in, N]
    # to calculate swapped version of kernel [c_in, c_out].
    bias_zero = np.zeros_like(self.bias)

    conv2d_cpu(self.input.transpose(1, 0, 2, 3), d_output.transpose(1, 0, 2, 3), bias_zero, self.d_kernel.transpose(1, 0, 2, 3), self.stride, self.padding, self.dilation)

    # d(input) = conv(d_out, flipped180(kernelT))
    d_input = np.zeros_like(self.input)
    flipped_kernel = np.zeros_like(self.kernel)

    flipKernel_cpu(self.kernel, flipped_kernel)
    flipped_kernel = flipped_kernel.transpose(1, 0, 2, 3)
    conv2d_cpu(d_output, flipped_kernel, bias_zero, d_input, self.stride, self.padding, self.dilation)
    return d_input

  @classmethod
  def class_name(cls):
    return "NNConv2D"

  def save_pickle(self):
    return {
        'class': self.class_name(),
        'in_channels': self.in_channels,
        'out_channels': self.out_channels,
        'kernel_size': self.kernel_size,
        'stride': self.stride,
        'padding': self.padding,
        'dilation': self.dilation,

        'kernel': self.kernel,
        'bias': self.bias,
        'd_kernel': self.d_kernel,
        'd_bias': self.d_bias
    }

  @classmethod
  def load_pickle(cls, data):
    obj = cls(data['in_channels'], data['out_channels'], data['kernel_size'], data['stride'], data['padding'], data['dilation'])
    obj.kernel = data['kernel']
    obj.bias = data['bias']
    obj.d_kernel = data['d_kernel']
    obj.d_bias = data['d_bias']
    return obj

In [11]:
import torch
import torch.nn.functional as F
import numpy as np
import pickle

# Settings
N, C_in, C_out = 2, 3, 4
H, W = 6, 6
K = 3
stride = 1
padding = 1
dilation = 1

# PyTorch layer
x_torch = torch.randn(N, C_in, H, W, dtype=torch.float32, requires_grad=True)
conv = torch.nn.Conv2d(C_in, C_out, K, stride=stride, padding=padding, dilation=dilation, bias=True)
with torch.no_grad():
    conv.weight[:] = torch.randn_like(conv.weight)
    conv.bias[:] = torch.randn_like(conv.bias)

# Forward + backward in PyTorch
out_torch = conv(x_torch)
loss_torch = out_torch.sum()
loss_torch.backward()

# Save PyTorch grads
d_input_torch = x_torch.grad.detach().numpy()
d_weight_torch = conv.weight.grad.detach().numpy()
d_bias_torch = conv.bias.grad.detach().numpy()

# NumPy equivalents
x_numpy = x_torch.detach().numpy()
weight_numpy = conv.weight.detach().numpy()
bias_numpy = conv.bias.detach().numpy()
d_output_numpy = np.ones_like(out_torch.detach().numpy())  # since loss was .sum()

# Use your NumPy layer
conv_np = NNConv2D(C_in, C_out, K, stride=stride, padding=padding, dilation=dilation)
conv_np.kernel = weight_numpy.copy()
conv_np.bias = bias_numpy.copy()

# Forward pass
_ = conv_np.forward(x_numpy)

# Backward pass
d_input_numpy = conv_np.backward(d_output_numpy)

# Compare
print("Grad Input close?  ", np.allclose(d_input_numpy, d_input_torch, atol=1e-5))
print("Grad Weight close? ", np.allclose(conv_np.d_kernel, d_weight_torch, atol=1e-5))
print("Grad Bias close?   ", np.allclose(conv_np.d_bias, d_bias_torch, atol=1e-5))

# Optionally: print max diff
print("Max d_input diff  :", np.abs(d_input_numpy - d_input_torch).max())
print("Max d_kernel diff :", np.abs(conv_np.d_kernel - d_weight_torch).max())
print("Max d_bias diff   :", np.abs(conv_np.d_bias - d_bias_torch).max())

with(open('data.pkl', 'wb')) as f:
    pickle.dump(conv_np.save_pickle(), f)

new_conv_np = NNConv2D.load_pickle(pickle.load(open('data.pkl', 'rb')))
print("Grad Weight close? ", np.allclose(new_conv_np.d_kernel, d_weight_torch, atol=1e-5))
print("Grad Bias close?   ", np.allclose(new_conv_np.d_bias, d_bias_torch, atol=1e-5))

Grad Input close?   True
Grad Weight close?  True
Grad Bias close?    True
Max d_input diff  : 9.536743e-07
Max d_kernel diff : 1.4305115e-06
Max d_bias diff   : 0.0
Grad Weight close?  True
Grad Bias close?    True
